[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Section_4_on_policy_control_complete.ipynb)

<div style="text-align:center">
    <h1>
        On-policy Monte Carlo Control
    </h1>
</div>
<br>

<div style="text-align:center">
    <p>
        In this notebook we are going to implement one of the two major strategies that exist when learning a policy by interacting with the environment, called on-policy learning. The agent will perform the task from start to finish and based on the sample experience generated, update its estimates of the q-values of each state-action pair $Q(s,a)$.
    </p>
</div>

<br>

In [ ]:
# === Setup: load the shared course modules =====================================
# The long environment-definition cell has been moved into two files that live
# next to this notebook: `envs.py` (the Maze environment) and `utils.py` (all the
# plotting / evaluation helpers). That keeps every lesson focused on the algorithm.
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('envs.py'):
    # On Google Colab there are no local files, so install the pinned gym version
    # and download the two helper modules from the course repository.
    !pip install -qq gym==0.23.0 pygame
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/envs.py
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils.py

from envs import Maze              # The 5x5 grid-world environment.
from utils import test_agent, plot_policy, plot_action_values


## Import the necessary software libraries:

In [ ]:
# NumPy gives us fast arrays for the Q-table and quick random choices.
import numpy as np
# Matplotlib lets us draw the maze, the Q-values and the learned policy.
import matplotlib.pyplot as plt

## Initialize the environment

In [ ]:
env = Maze()

In [ ]:
# Ask the environment for a picture (an RGB image) of the current maze.
frame = env.render(mode='rgb_array')
# Hide the x/y axis ticks so we just see the maze cleanly.
plt.axis('off')
# Display that image inside the notebook.
plt.imshow(frame)

In [ ]:
# Shape of the state space: how many rows and columns the grid has.
print(f"Observation space shape: {env.observation_space.nvec}")
# How many distinct actions the agent can take (up, down, left, right).
print(f"Number of actions: {env.action_space.n}")

## Define value table $Q(s, a)$

#### Create the $Q(s, a)$ table

In [ ]:
# Create the Q-table: one value for every (row, column, action) combination.
# It is 5 x 5 (the grid) x 4 (the four actions), and we start every value at 0.
action_values = np.zeros(shape=(5, 5, 4))

#### Plot Q(s, a)

In [ ]:
# Draw the current Q-values so we can see them (all zero before any learning).
plot_action_values(action_values)

## Define the policy $\pi(s)$

#### Create the policy $\pi(s)$

In [ ]:
# An epsilon-greedy policy: mostly act greedily, but sometimes explore at random.
def policy(state, epsilon=0.):
    # With probability epsilon, ignore what we know and pick a random action (explore).
    if np.random.random() < epsilon:
        return np.random.randint(4)
    else:
        # Otherwise look up the four Q-values for this state.
        av = action_values[state]
        # Pick the action with the highest value; if several tie, choose among them at random.
        return np.random.choice(np.flatnonzero(av == av.max()))

#### Test the policy with state (0, 0)

In [ ]:
# Ask the policy what it would do in the top-left corner state (0, 0).
action = policy((0,0))
# Print the chosen action so we can check the policy runs.
print(f"Action taken in state (0,0): {action}")

#### Plot the policy

In [ ]:
# Draw an arrow in each cell showing the action the greedy policy currently prefers.
plot_policy(action_values, frame)

## Implement the algorithm

### What is *control* in reinforcement learning?

So far we only had to *follow* a fixed plan. **Control** means something harder: the
agent has to **discover a good plan by itself**, just by trying actions and seeing what
happens. It keeps a table of values `Q(s, a)` -- "how good is taking action `a` in
state `s`?" -- and slowly improves those numbers from experience.

### What does *on-policy* mean?

There are two families of control methods, and they differ in one simple question:
*does the agent learn about the very same behaviour it is using to act?*

- **On-policy** (this notebook): the agent **learns about the same policy it follows**.
  It explores a little, and it improves *that exact exploring policy*.
- **Off-policy** (next notebook): the agent can act with one policy but learn about a
  different one.

### Why epsilon-greedy?

If the agent always picked the action it currently thinks is best (*greedy*), it would
never try anything new and could get stuck with a bad habit. To avoid that we use an
**epsilon-greedy** policy:

- with probability **(1 - epsilon)** take the best-known action (**exploit**),
- with probability **epsilon** take a random action (**explore**).

A small `epsilon` (here 0.2) keeps just enough randomness to keep discovering the maze.

### Monte Carlo control, step by step

"Monte Carlo" simply means **we learn from complete episodes**, using the *actual*
returns we observed -- no guessing about the future.

1. **Play a full episode** from start to goal, recording every `(state, action, reward)`.
2. **Walk backwards** through the episode and compute the **return** `G` at each step
   (the total discounted reward that followed from there):
   `G = reward_t + gamma * G`.
   The discount factor **gamma** (here 0.99) makes rewards that arrive sooner count for
   a little more than far-away ones.
3. For each `(state, action)` pair, **average all the returns** ever seen for it. That
   average becomes the new `Q(s, a)`.

Because the policy always reads from this `Q` table, improving the numbers automatically
improves the policy -- that is the on-policy loop in action.

In [ ]:
# On-policy first-visit-style Monte Carlo control with an epsilon-greedy policy.
def on_policy_mc_control(policy, action_values, episodes, gamma=0.99, epsilon=0.2):

    # Dictionary that stores every return seen for each (state, action) pair.
    sa_returns = {}

    # Repeat the whole learn-from-one-episode process many times.
    for episode in range(1, episodes+1):
        # Start a fresh episode and get the first state.
        state = env.reset()
        # 'done' becomes True when the agent reaches the goal (episode ends).
        done = False
        # We record the full trajectory (state, action, reward) of this episode here.
        transitions = []

        # Play one complete episode from start to finish.
        while not done:
            # Choose an action with the SAME policy we are improving (that is "on-policy").
            action = policy(state, epsilon)
            # Take the action; the environment returns the next state and the reward.
            next_state, reward, done, _ = env.step(action)
            # Save this step so we can learn from it after the episode ends.
            transitions.append([state, action, reward])
            # Move to the next state and continue.
            state = next_state

        # G is the return (sum of discounted future rewards); start it at 0.
        G = 0
        # Walk through the episode backwards so each return builds on the next step's.
        for state_t, action_t, reward_t in reversed(transitions):
            # Add this step's reward to the discounted return from later steps.
            G = reward_t + gamma * G

            # First time we see this (state, action) pair, create an empty list for it.
            if not (state_t, action_t) in sa_returns:
                sa_returns[(state_t, action_t)] = []
            # Record the return we just computed for this (state, action) pair.
            sa_returns[(state_t, action_t)].append(G)
            # Set the Q-value to the AVERAGE of all returns ever seen for this pair.
            action_values[state_t][action_t] = np.mean(sa_returns[(state_t, action_t)])

In [ ]:
# Train the agent by running the algorithm for 10,000 episodes.
on_policy_mc_control(policy, action_values, episodes=10000)

## Show results

#### Show resulting value table $Q(s, a)$

In [ ]:
# Plot the learned Q-values; useful states near the goal should now look better.
plot_action_values(action_values)

#### Show resulting policy $\pi(\cdot|s)$

In [ ]:
# Draw the final greedy policy; arrows should now point toward the goal.
plot_policy(action_values, frame)

#### Test the resulting agent

In [ ]:
# Run the greedy agent in the maze and watch it solve the task.
test_agent(env, policy)

## Resources

[[1] Reinforcement Learning: An Introduction. Ch. 4: Dynamic Programming](https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf)